In [ ]:
from cemp_software_settings import load_and_apply_settings
result = load_and_apply_settings()
print(result)

In [ ]:
import resource
import os

os.environ["OMP_STACKSIZE"] = "256M"
os.environ["KMP_STACKSIZE"] = "256M"


resource.setrlimit(resource.RLIMIT_STACK, (resource.RLIM_INFINITY, resource.RLIM_INFINITY))

In [ ]:
import re
import glob
import os
import subprocess
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
from openbabel import openbabel, pybel
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AddHs
from rdkit.Chem.rdmolfiles import MolToSmiles
from openbabel import openbabel, pybel
from rdkit.Chem import RemoveHs
import tempfile
from scipy.signal import argrelextrema
import numpy as np
import matplotlib.pyplot as plt
import MDAnalysis as mda
import xdrlib
from MDAnalysis.lib.util import NamedStream
import random
import time
import imageio
from concurrent.futures import ProcessPoolExecutor, as_completed
from concurrent.futures import ThreadPoolExecutor
import threading
import MDAnalysis as mda
import xdrlib
from MDAnalysis.lib.util import NamedStream
import signal
from scipy.signal import medfilt
import shutil
import zipfile
from numpy.fft import fftn, fftfreq
import matplotlib.pyplot as plt

In [ ]:


import json
import os

def load_radius(filename="rdf_radius.json"):
    with open(os.path.join(os.getcwd(), filename), "r", encoding="utf-8") as f:
        data = json.load(f)
    return data.get("rdf_radius")

rdf_radius = load_radius()                     
print(rdf_radius)  

In [ ]:
import MDAnalysis as mda
import pandas as pd


df = pd.read_excel('System.xlsx')



center_name = df['center atom'].iloc[0]


u = mda.Universe('prod_NVT.tpr','prod_NVT.xtc',indices='index.ndx')


In [ ]:

def get_group_indices(group_name, ndx_filename='index.ndx'):
    
    with open(ndx_filename, 'r') as ndx_file:
        in_group = False
        indices = []
        for line in ndx_file:
            
            if line.startswith('['):
                
                name_in_brackets = line[1:line.find(']')].strip()
                
                in_group = (group_name == name_in_brackets)
                continue
            
            if in_group and line.strip():
                indices.extend(map(int, line.split()))
        return indices


In [ ]:

ion_dict = {}
ion_indices = get_group_indices(center_name)
ion_dict[center_name] = ion_indices

In [ ]:
ion_dict

In [ ]:
import os
import shutil
import subprocess


def extract_total_coordination_environments(ion_dict,
                                            center_name,
                                            rdf_radius,
                                            structure_file='prod_NVT.gro',
                                            xtc_file='prod_NVT.xtc',
                                            N_frame: int = 10):
    

    
    
    if center_name not in ion_dict:
        print("Public message removed for release.")
        return

    center_indices = ion_dict[center_name]
    if not center_indices:
        print("Public message removed for release.")
        return

    
    selected_indices_vmd = [idx - 1 for idx in center_indices]

    
    if not os.path.isfile(structure_file):
        print("Public message removed for release.")
        return
    if not os.path.isfile(xtc_file):
        print("Public message removed for release.")
        return

    
    output_dir = 'total_coordination_structure'
    os.makedirs(output_dir, exist_ok=True)
    abs_output_dir = os.path.abspath(output_dir)

    
    for idx in selected_indices_vmd:
        print("Public message removed for release.")

    
    
    indices_list_str = " ".join(str(i) for i in selected_indices_vmd)

    tcl_script = 

    
    tcl_filename = 'extract_coordination_xtc.tcl'
    with open(tcl_filename, 'w') as f:
        f.write(tcl_script)

    try:
        vmd_executable = 'vmd'
        if not shutil.which(vmd_executable):
            print("Public message removed for release.")
            return

        print("Public message removed for release.")
        subprocess.run([vmd_executable, '-dispdev', 'none', '-e', tcl_filename], check=True)
        print("Public message removed for release.")
    except subprocess.CalledProcessError as e:
        print("Public message removed for release.")
    finally:
        if os.path.exists(tcl_filename):
            os.remove(tcl_filename)

In [ ]:
extract_total_coordination_environments(ion_dict, center_name, rdf_radius)

In [ ]:
import os
import re
import pandas as pd

def parse_itp_files(df_system):
    

    
    itp_atom_type_dict = {}

    
    header_pattern = re.compile(r'^;\s+Index\s+type\s+residue\s+resname\s+atom\s+cgnr\s+charge\s+mass')

    
    
    data_pattern = re.compile(
        r'^\s*(\d+)\s+'        
        r'(\S+)\s+'            
        r'(\d+)\s+'            
        r'(\S+)\s+'            
        r'(\S+)\s+'            
        r'(\d+)\s+'            
        r'([\-\+\d\.]+)\s+'    
        r'([\-\+\d\.]+)\s*$'   
    )

    
    for idx, row in df_system.iterrows():
        
        name = row['Name']

        
        itp_filename = f"{name}.itp"
        if not os.path.isfile(itp_filename):
            
            print("Public message removed for release.")
            continue

        
        itp_atom_type_list = []

        
        with open(itp_filename, 'r', encoding='utf-8') as f:
            lines = f.readlines()

        
        header_index = None
        for i, line in enumerate(lines):
            if header_pattern.search(line):
                header_index = i
                break

        if header_index is None:
            
            print("Public message removed for release.")
            continue

        
        start_line_index = header_index + 1

        for line in lines[start_line_index:]:
            
            match_data = data_pattern.match(line)
            if match_data:
                atom_name = match_data.group(5)  
                atom_name = atom_name[:4]  
                itp_atom_type_list.append(atom_name)
            else:
                
                
                break

        
        itp_atom_type_dict[name] = itp_atom_type_list

    
    return itp_atom_type_dict


In [ ]:
df_system = pd.read_excel("System.xlsx")


itp_atom_type_dict = parse_itp_files(df_system)


for k, v in itp_atom_type_dict.items():
    print("Public message removed for release.")

In [ ]:
import os
import re
from collections import defaultdict

def parse_coordination_mol2_files(itp_atom_type_dict, coordination_structure_to_compute_path="total_coordination_structure"):
    

    
    if coordination_structure_to_compute_path is None:
        coordination_structure_to_compute_path = os.path.join(os.getcwd(), "coordination_structure_to_compute")

    
    if not os.path.isdir(coordination_structure_to_compute_path):
        print("Public message removed for release.")
        return {}

    
    coordination_mol_dict = {}

    
    mol2_files = [f for f in os.listdir(coordination_structure_to_compute_path) 
                  if f.endswith(".mol2")]

    
    if not mol2_files:
        print("Public message removed for release.")
        return {}

    
    
    
    reverse_map = {}
    for dict_key, dict_values in itp_atom_type_dict.items():
        for val in dict_values:
            reverse_map[val] = dict_key

    
    atom_block_pattern = re.compile(r'^@<TRIPOS>ATOM')

    
    for mol2_file in mol2_files:
        
        coordination_structure_name = os.path.splitext(mol2_file)[0]

        
        mol2_path = os.path.join(coordination_structure_to_compute_path, mol2_file)
        with open(mol2_path, "r", encoding="utf-8") as f:
            lines = f.readlines()

        
        coordination_structure_atom_type_list = []

        
        atom_block_start = None
        for i, line in enumerate(lines):
            if atom_block_pattern.match(line.strip()):
                atom_block_start = i
                break

        
        if atom_block_start is None:
            print("Public message removed for release.")
            continue

        
        i = atom_block_start + 1
        while i < len(lines):
            current_line = lines[i].strip()
            
            if not current_line or current_line.startswith('@<TRIPOS>'):
                break

            
            
            parts = current_line.split()
            if len(parts) < 2:
                
                break
            
            atom_type = parts[1]
            coordination_structure_atom_type_list.append(atom_type)

            i += 1

        
        

        
        
        composition_count = defaultdict(int)
        for atype in coordination_structure_atom_type_list:
            
            if atype in reverse_map:
                composition_key = reverse_map[atype]
                composition_count[composition_key] += 1
            else:
                
                
                print("Public message removed for release.")
                continue
        
        
        
        
        coordination_mol_number_list = {}
        for comp_key, alias_list in itp_atom_type_dict.items():
            denom = len(alias_list) if len(alias_list) > 0 else 1  
            raw_count = composition_count.get(comp_key, 0)         
            coordination_mol_number_list[comp_key] = raw_count
            
        
        coordination_mol_dict[coordination_structure_name] = coordination_mol_number_list

    
    return coordination_mol_dict




In [ ]:

coordination_mol_dict = parse_coordination_mol2_files(itp_atom_type_dict)


for structure_name, composition in coordination_mol_dict.items():
    print("Public message removed for release.")

In [ ]:
coordination_mol_dict

In [ ]:

from __future__ import annotations
from typing import Dict, Any, List, Tuple
from collections import defaultdict
from hashlib import sha256
from datetime import datetime
from pathlib import Path
import json
import pandas as pd


def summarize_coordination_environments(coordination_mol_dict: Dict[str, Dict[str, Any]],
                                        df: pd.DataFrame) -> str:
    
    
    if not isinstance(df, pd.DataFrame) or "Name" not in df.columns:
        raise ValueError("Public message removed for release.")

    
    species_order: List[str] = [str(x) for x in df["Name"].tolist()]

    
    unknown_counts = defaultdict(int)
    unknown_records: List[Dict[str, Any]] = []

    
    groups: Dict[str, Dict[str, Any]] = {}

    
    def _standardize(comp: Dict[str, Any]) -> Tuple[
        Dict[str, Any],           
        Dict[str, Any],           
        List[Any],                
        str, str,                 
        str, str                  
    ]:
        
        def _coerce_number(v: Any) -> Any:
            if isinstance(v, bool):
                return int(v)
            if isinstance(v, (int, float)):
                return int(v) if float(v).is_integer() else float(v)
            if isinstance(v, str):
                s = v.strip()
                if s == "":
                    return 0
                try:
                    f = float(s)
                    return int(f) if f.is_integer() else f
                except Exception:
                    return 0
            return 0

        
        full = {name: _coerce_number(comp.get(name, 0)) for name in species_order}
        vector = [full[name] for name in species_order]

        
        reduced_items = sorted(((k, v) for k, v in full.items() if v != 0), key=lambda kv: kv[0])
        reduced = {k: v for k, v in reduced_items}

        
        def _num2str(x: Any) -> str:
            return str(int(x)) if isinstance(x, float) and x.is_integer() else str(x)

        name_sorted_repr = "|".join([f"{k}={_num2str(v)}" for k, v in reduced_items])
        name_sorted_hash = sha256(name_sorted_repr.encode("utf-8")).hexdigest()

        
        df_order_vector_repr = json.dumps(vector, ensure_ascii=False, separators=(",", ":"))
        df_order_vector_hash = sha256(df_order_vector_repr.encode("utf-8")).hexdigest()

        return reduced, full, vector, name_sorted_repr, name_sorted_hash, df_order_vector_repr, df_order_vector_hash

    
    for key, comp in coordination_mol_dict.items():
        if not isinstance(comp, dict):
            raise TypeError("Public message removed for release.")

        
        extra_keys = [k for k in comp.keys() if k not in species_order]
        if extra_keys:
            for k in extra_keys:
                unknown_counts[str(k)] += 1
            unknown_records.append({"coordination_key": key, "unknown_keys": [str(k) for k in extra_keys]})

        reduced, full, vector, name_repr, name_hash, vec_repr, vec_hash = _standardize(comp)

        
        if name_hash not in groups:
            groups[name_hash] = {
                "composition": reduced,                
                "composition_full": full,              
                "signature": {
                    "name_sorted_repr": name_repr,
                    "name_sorted_hash": name_hash,
                    "df_order_vector_repr": vec_repr,
                    "df_order_vector_hash": vec_hash
                },
                "members": [],                         
                "members_detail": []                   
            }
        groups[name_hash]["members"].append(key)
        groups[name_hash]["members_detail"].append({"coordination_key": key, "value": comp})

    
    
    sorted_items = sorted(groups.items(),
                          key=lambda kv: (-len(kv[1]["members"]), kv[1]["signature"]["name_sorted_repr"]))

    environments = []
    for idx, (env_key, payload) in enumerate(sorted_items, start=1):
        environments.append({
            "env_id": idx,
            "count": len(payload["members"]),
            "composition": payload["composition"],
            "composition_full": payload["composition_full"],
            "signature": payload["signature"],
            "members": sorted(payload["members"]),
            "members_detail": payload["members_detail"]  
        })

    result = {
        "schema_version": 1,
        "created_at": datetime.now().isoformat(timespec="seconds"),
        "summary": {
            "total_entries": len(coordination_mol_dict),
            "unique_environments": len(environments),
            "df_name_order": species_order
        },
        "environments": environments,
        "validation": {
            "unknown_keys_total": int(sum(unknown_counts.values())),
            "unknown_keys_by_name": dict(sorted(unknown_counts.items())),
            "entries_with_unknown_keys": unknown_records
        }
    }

    
    out_name = "coordination_env_summary.json"
    out_path = Path.cwd() / out_name
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)

    return str(out_path.resolve())

In [ ]:
summarize_coordination_environments(coordination_mol_dict, df)

In [ ]:

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm

def plot_coordination_polar_from_json(
    json_path: str,
    df: pd.DataFrame,
    output: bool = True,
    figsize=(4, 4),
    figname: str = "coordination_polar.png",
    dpi: int = 600,
    show_percent: bool = True,
    gamma: float = 0.5,              
    cmap_name: str = "Blues",        
    cmap_reverse: bool = False,      
    q_low: float = 5.0,              
    q_high: float = 95.0,            
    color_span: tuple = (0.35, 0.80) 
):
    
    
    if not isinstance(df, pd.DataFrame) or "Name" not in df.columns:
        raise ValueError("Public message removed for release.")
    species_order = [str(x) for x in df["Name"].tolist()]

    
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    env_list = data.get("environments", [])
    if not env_list:
        raise ValueError("Public message removed for release.")

    
    df_name_order_from_json = data.get("df_name_order", None)
    if df_name_order_from_json is None:
        df_name_order_from_json = data.get("summary", {}).get("df_name_order", species_order)
    title_text = "_".join(map(str, df_name_order_from_json))

    
    counts, labels = [], []
    for env in env_list:
        c = int(env.get("count", len(env.get("members", []))))
        counts.append(c)

        comp_full = env.get("composition_full", None)
        if comp_full is None:
            comp = env.get("composition", {})
            comp_full = {k: 0 for k in species_order}
            for k, v in comp.items():
                if k in comp_full:
                    comp_full[k] = v
        vect = [comp_full.get(name, 0) for name in species_order]

        
        def _fmt_num(x):
            try:
                xf = float(x)
                return str(int(round(xf))) if abs(xf - round(xf)) < 1e-9 else str(xf)
            except Exception:
                return str(x)
        labels.append("-".join(_fmt_num(v) for v in vect))

    total = data.get("summary", {}).get("total_entries", sum(counts))
    total = max(int(total), 1)
    percents = np.array([100.0 * c / total for c in counts], dtype=float)

    
    plt.rcParams.update({
        "font.family": "Arial",
        "font.size": 7,
        "axes.linewidth": 0.75,
        "figure.facecolor": "white",
        "savefig.facecolor": "white"
    })

    
    N = len(percents)
    theta = np.linspace(0, 2*np.pi, N, endpoint=False)
    width = 2*np.pi / max(N, 1) * 0.88
    h_max = max(percents) if N > 0 else 1.0
    r_inner = 0.60 * h_max

    
    if h_max > 0:
        norm_h = percents / h_max
        height_vis = (norm_h ** gamma) * h_max
    else:
        height_vis = percents.copy()

    
    cmap = cm.get_cmap(cmap_name)
    if cmap_reverse:
        try:
            cmap = cm.get_cmap(cmap_name + "_r")
        except ValueError:
            cmap = cmap.reversed()
    if np.allclose(percents.max(), percents.min()):
        vals = np.full(N, 0.5)  
    else:
        lo = np.percentile(percents, q_low)
        hi = np.percentile(percents, q_high)
        if hi - lo < 1e-12:
            lo, hi = percents.min(), percents.max()
        vals = np.clip((percents - lo) / max(hi - lo, 1e-12), 0, 1)

    
    low_span, high_span = color_span
    low_span = max(0.0, min(low_span, 1.0))
    high_span = max(0.0, min(high_span, 1.0))
    if high_span <= low_span:
        low_span, high_span = 0.35, 0.80  
    vals = low_span + vals * (high_span - low_span)
    colors = [cmap(v) for v in vals]

    
    fig = plt.figure(figsize=figsize)
    ax = fig.add_subplot(111, polar=True)

    
    fig.suptitle(title_text, y=0.98)

    
    ax.bar(theta, height_vis, width=width, bottom=r_inner,
           linewidth=0, edgecolor="none", color=colors)

    
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(False)
    ax.spines["polar"].set_visible(False)

    
    center_text = "_".join(species_order)
    ax.text(0.0, 0.0, center_text, ha="center", va="center")

    
    offset = 0.03 * h_max
    for ang, h, lab in zip(theta, height_vis, labels):
        deg = np.degrees(ang)
        rotation = deg - 90
        ha = "left"
        if 90 < deg < 270:
            rotation += 180
            ha = "right"
        ax.text(ang, r_inner + h + offset, lab,
                rotation=rotation, rotation_mode="anchor",
                ha=ha, va="center")

    
    if show_percent:
        for ang, h, p in zip(theta, height_vis, percents):
            ax.text(ang, r_inner + 0.5 * h, f"{p:.1f}%", ha="center", va="center")

    
    if output:
        fig.savefig(figname, dpi=dpi, bbox_inches="tight")
    else:
        plt.show()

In [ ]:



plot_coordination_polar_from_json("coordination_env_summary.json",
                                  df,
                                  figname = "coordination_polar.png",
                                  gamma = 0.05,
                                  color_span = (0.2, 0.80)) 


In [ ]:
import zipfile
import shutil


def collect_and_compress_files(source_dir):
    
    target_extensions = ('.tif', '.gif', '.mp4', '.mol2', '.png', '.json')

    
    all_results_dir = os.path.join(source_dir, 'all_results')
    if not os.path.exists(all_results_dir):
        os.makedirs(all_results_dir)
    
    
    exact_files = ["prod_NVT.xtc", "prod_NVT.gro", "index.ndx"]

    
    for file in os.listdir(source_dir):
        source_file = os.path.join(source_dir, file)
        
        
        pattern = (
            r'^.*_(?:rdf_cn|rdf|msd)\.xvg$'    
            r'|^hbnum_.*\.xvg$'               
            r'|^hblife_.*\.xvg$'              
        )
        
        if os.path.isfile(source_file) and file in exact_files:
            destination_file = os.path.join(all_results_dir, file)
            if os.path.abspath(source_file) != os.path.abspath(destination_file):
                shutil.copy2(source_file, destination_file)
            continue  

        if os.path.isfile(source_file) and (file.endswith(target_extensions) or re.match(pattern, file)):
            destination_file = os.path.join(all_results_dir, file)
            
            if os.path.abspath(source_file) != os.path.abspath(destination_file):
                shutil.copy2(source_file, destination_file)
        
        
        elif os.path.isdir(source_file) and file == 'total_coordination_structure':
            target_gaussian_folder = os.path.join(all_results_dir, 'total_coordination_structure')
            
            if not os.path.exists(target_gaussian_folder):
                shutil.copytree(source_file, target_gaussian_folder)
        

    
    zip_filename = os.path.join(source_dir, 'all_results.zip')
    with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(all_results_dir):
            for file in files:
                
                file_path = os.path.join(root, file)
                zipf.write(file_path, os.path.relpath(file_path, all_results_dir))
    
    print("Public message removed for release.")


source_dir = '.'  
collect_and_compress_files(source_dir)